# SZ cerebellum: smoke-test results from newly simulated NIO/HDF5

Use this notebook **after** running `reproducibility/scripts/replay.py` from the TEST repository. By default, the replay tests `k=1.000`, `0.760`, `0.600` in each of six cases. The analysis computes **raw, population-averaged firing rates** directly from the recorded spikes, divided by the cell count of the corresponding reconstructed HDF5 and the simulation duration.

The eight NIO files for one simulation are MPI partitions, **not independent replicates**. This notebook is a diagnostic and does not reproduce the full manuscript's kernel-smoothed response, Kase alignment, transfer entropy, or other multi-stage figure statistics. Full paper figures require the complete sweep and the historical analysis intermediate files.

In [ ]:
from pathlib import Path
import os

candidates=[Path.cwd(),*Path.cwd().parents,Path.cwd()/".."]
repo=next((p.resolve() for p in candidates if (p/"reproducibility/scripts/analyze_quick_results.py").is_file()),None)
if repo is None:
    raise RuntimeError("Launch Jupyter from the SZcerebellumModel-TEST repository root or its notebooks/ directory")
outputs=Path(os.getenv("SZ_OUTPUT_DIR",str(Path.home()/"SZ_replay_outputs"))).expanduser().resolve()
report=outputs/"analysis_smoke"
print("Repository:",repo)
print("Simulation outputs:",outputs)
print("Analysis results:",report)

In [ ]:
import sys
import subprocess

cmd=[sys.executable,str(repo/"reproducibility/scripts/analyze_quick_results.py"),"--input",str(outputs),"--output",str(report),"--strict"]
result=subprocess.run(cmd,check=True,cwd=str(repo))
print("Analysis finished; return code:",result.returncode)

In [ ]:
import pandas as pd
rows=pd.read_csv(report/"smoke_firing_rates.csv")
display(rows.loc[rows.population.isin(["purkinje","dcn_p","mossy_fibers"]),["case","k","population","cell_count","firing_rate_hz"]].sort_values(["case","k","population"],ascending=[True,False,True]).reset_index(drop=True))

In [ ]:
from IPython.display import Image,display
for name in ("autor_smoke_firing.png","basal_smoke_firing.png","evoked_smoke_firing.png"):
    image_path=report/name
    if image_path.is_file():
        display(Image(filename=str(image_path)))

## Interpreting the results

Each protocol should have **three data points** per HR/NR curve. This is a **pipeline check**, not a statistical trend analysis. For publication-level comparisons, run the complete 16-point sweep with `replay.py --full` and use the historical post-processing methodology and notebooks, checking all required intermediate files.

See [the notebooks README](README.md) for the three original archived notebooks and their dependencies.